# Ćwiczenie 0: Sprawdź swoje środowisko

Ten notatnik nie uczy uczenia maszynowego. Ma jedno zadanie: **potwierdzić, że wszystko jest zainstalowane poprawnie**, zanim zaczniesz ćwiczenie 01.

Instrukcję instalacji znajdziesz w pliku [`00_INSTALACJA.md`](00_INSTALACJA.md).

## Jak z niego korzystać

Wybierz z menu **Run → Run All Cells**. Każda komórka sprawdza jedną rzecz i wypisuje wynik:

- `[OK]` - działa,
- `[BŁĄD]` - trzeba naprawić (pod spodem znajdziesz wskazówkę, jak).

Jeśli wszystkie sprawdzenia przejdą, ostatnia komórka wypisze potwierdzenie i możesz przechodzić dalej.

## 1. Wersja Pythona

Potrzebujemy Pythona **3.10 lub nowszego**. Starsze wersje nie obsługują części składni używanej w bibliotekach, z których korzystamy.

In [ ]:
import sys

wersja = sys.version_info
print(f"Twoja wersja Pythona: {wersja.major}.{wersja.minor}.{wersja.micro}")
print(f"Interpreter: {sys.executable}")
print()

if (wersja.major, wersja.minor) >= (3, 10):
    print("[OK] Wersja Pythona jest wystarczająca.")
else:
    print("[BŁĄD] Potrzebujesz Pythona 3.10 lub nowszego.")
    print("       Zainstaluj nowszą wersję zgodnie z instrukcją w 00_INSTALACJA.md.")

> **Zwróć uwagę na ścieżkę interpretera** wypisaną wyżej. Jeśli pracujesz w środowisku wirtualnym, powinna prowadzić do katalogu `.venv` albo `ml-kurs`, a **nie** do systemowego Pythona. Jeśli prowadzi gdzie indziej, JupyterLab prawdopodobnie działa na złym środowisku - wróć do instrukcji instalacji.

## 2. Wymagane pakiety

Sprawdzamy, czy wszystkie biblioteki są zainstalowane i w jakich wersjach.

In [ ]:
wymagane = {
    "numpy":        "obliczenia na tablicach liczb",
    "pandas":       "wczytywanie i obróbka danych tabelarycznych",
    "matplotlib":   "wykresy",
    "sklearn":      "modele uczenia maszynowego (pakiet scikit-learn)",
    "joblib":       "zapis modelu do pliku (ćwiczenie 10)",
}

import importlib

braki = []
for nazwa, opis in wymagane.items():
    try:
        modul = importlib.import_module(nazwa)
        wersja = getattr(modul, "__version__", "wersja nieznana")
        print(f"[OK]    {nazwa:12s} {wersja:10s}  - {opis}")
    except ImportError:
        print(f"[BŁĄD]  {nazwa:12s} {'BRAK':10s}  - {opis}")
        braki.append(nazwa)

print()
if braki:
    print("Brakuje pakietów:", ", ".join(braki))
    print("Uruchom w terminalu (przy aktywnym środowisku):")
    print("    pip install -r requirements.txt")
else:
    print("[OK] Wszystkie wymagane pakiety są zainstalowane.")

## 3. Dostęp do danych

Ćwiczenia korzystają z pliku `dane/diabetes.csv`. Ta komórka sprawdza, czy notatnik go widzi.

To najczęstsze źródło błędów na starcie: jeśli JupyterLab uruchomiono z innego katalogu, ścieżka względna nie zadziała.

In [ ]:
import os
import pandas as pd

print("Katalog roboczy notatnika:", os.getcwd())
print()

sciezka = "dane/diabetes.csv"

if os.path.exists(sciezka):
    dane = pd.read_csv(sciezka)
    print(f"[OK] Wczytano dane: {dane.shape[0]} wierszy, {dane.shape[1]} kolumn.")
    print()
    print("Kolumny:", ", ".join(dane.columns))
else:
    print(f"[BŁĄD] Nie znaleziono pliku: {sciezka}")
    print("       JupyterLab musi być uruchomiony z katalogu 'cwiczenia-ml'.")
    print("       Zamknij go (Ctrl+C w terminalu), przejdź do właściwego katalogu")
    print("       poleceniem 'cd', i uruchom 'jupyter lab' ponownie.")

## 4. Czy wykresy się rysują?

Wykresy w notatniku pojawiają się pod komórką, która je tworzy. Jeśli poniżej zobaczysz wykres słupkowy - wszystko działa.

In [ ]:
import matplotlib.pyplot as plt

if os.path.exists(sciezka):
    liczebnosc = dane["Diabetic"].value_counts().sort_index()

    fig, ax = plt.subplots(figsize=(5, 3.5))
    ax.bar(["brak cukrzycy", "cukrzyca"], liczebnosc.values, color=["#4C72B0", "#C44E52"])
    ax.set_ylabel("liczba pacjentów")
    ax.set_title("Rozkład klas w zbiorze diabetes")
    for i, v in enumerate(liczebnosc.values):
        ax.text(i, v, f"{v}\n({v / len(dane):.1%})", ha="center", va="bottom")
    ax.set_ylim(0, liczebnosc.max() * 1.18)
    plt.tight_layout()
    plt.show()

    print("[OK] Jeśli widzisz wykres powyżej - matplotlib działa poprawnie.")
else:
    print("[POMINIĘTO] Najpierw napraw dostęp do danych (punkt 3).")

## 5. Czy scikit-learn potrafi wytrenować model?

Ostatnie sprawdzenie: trenujemy najprostszy możliwy model. Nie chodzi o wynik, tylko o to, czy cały łańcuch działa od początku do końca.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier

if os.path.exists(sciezka):
    X = dane.drop(columns=["PatientID", "Diabetic"])
    y = dane["Diabetic"]

    X_ucz, X_test, y_ucz, y_test = train_test_split(
        X, y, test_size=0.2, stratify=y, random_state=42
    )

    model = DecisionTreeClassifier(max_depth=3, random_state=42)
    model.fit(X_ucz, y_ucz)

    print(f"[OK] Model wytrenowany. Skuteczność na zbiorze testowym: {model.score(X_test, y_test):.1%}")
    print()
    print("Nie przejmuj się teraz tą liczbą - co ona znaczy i czy jest dobra,")
    print("wyjaśnimy w ćwiczeniu 01.")
else:
    print("[POMINIĘTO] Najpierw napraw dostęp do danych (punkt 3).")

## 6. Podsumowanie

In [ ]:
wszystko_ok = (
    (sys.version_info.major, sys.version_info.minor) >= (3, 10)
    and not braki
    and os.path.exists(sciezka)
)

print("=" * 58)
if wszystko_ok:
    print("  ŚRODOWISKO GOTOWE")
    print("=" * 58)
    print()
    print("  Możesz przejść do ćwiczenia 01_pierwszy_model.ipynb")
else:
    print("  ŚRODOWISKO WYMAGA POPRAWEK")
    print("=" * 58)
    print()
    print("  Przejrzyj komunikaty [BŁĄD] powyżej i zajrzyj do 00_INSTALACJA.md.")
    print("  Zgłoś problem prowadzącemu PRZED zajęciami.")
print()

---

## Co dalej

Środowisko masz gotowe - czas na właściwą pracę.

Zacznij od **[01_pierwszy_model.ipynb](01_pierwszy_model.ipynb)**: zbudujesz tam swój pierwszy model uczenia maszynowego i - co ważniejsze - dowiesz się, dlaczego ocena modelu jest trudniejsza, niż się wydaje.

> **Jedna rada na start**: jeśli kiedykolwiek notatnik zacznie się zachowywać dziwnie (zmienna ma nie tę wartość, co powinna; kod, który działał, przestał), użyj **Kernel → Restart Kernel and Run All Cells**. Notatnik pamięta stan ze wszystkich komórek uruchomionych wcześniej - również tych, które później zmieniłeś lub usunąłeś. Restart wymusza wykonanie wszystkiego od zera, w kolejności, w jakiej komórki są zapisane. To najszybszy sposób, żeby sprawdzić, czy Twój notatnik naprawdę działa.